# Sovereign bond futures from EMIR

Builds one DEVO table, `lab_prj_emir_ecb.hermesf_fut`, from the ECB EMIR trade state tables.
Euro area contracts (Eurex, ISIN list in `gov_fut.csv`) and US Treasury futures (CBOT, identified by venue and CFI).
One row per trade state and reporting day, with buyer and seller instead of reporter and direction.
Cleaning rules are applied inside the queries, see `cleaning_decisions.txt`.

In [ ]:
import pyodbc
import pandas as pd
import time
from datetime import datetime

cnxn = pyodbc.connect('DSN=Hermes_DSN', autocommit=True)
cursor = cnxn.cursor()

In [ ]:
# parameters
eurex_lei = ('529900LN3S50JPU47S06', '529900VIIHQECIS4ET47', '529900UT4DG0LG5R9O07')

prerefit_start, prerefit_end = '2018-01-01', '2024-04-26'
refit_start,    refit_end    = '2025-01-01', '2026-09-01'   # REFIT transition window (Apr to Dec 2024) excluded

schema = 'lab_prj_emir_ecb'                                 # where the output table goes
table = f'{schema}.hermesf_fut'
sector_table = 'xlab_ecb_prj_sftds_cb_common.pth_sector'

# euro area contract list (ISIN, country, expiration date)
euro_govs = pd.read_csv('gov_fut.csv')
euro_govs_isin = tuple(euro_govs['isin'].unique())
gov_fut_sql = '\n        UNION ALL '.join(
    f"SELECT '{r.isin}' AS isin, '{r.country}' AS country, '{r.expiration_date}' AS expiration_date"
    for r in euro_govs.itertuples()
)

## Query builder

In [ ]:
# column names that differ between the two reporting regimes
sources = {
    'prerefit': dict(table='crp_emir_ecb.emir_ecb_states_deduplicated',
                     direction='leg1_counterparty_side', country_suffix='',
                     isin='leg1_product_id', cfi='leg1_product_clssfctn',
                     notional='leg1_notional', ccy='leg1_notional_currency1',
                     price='leg1_price_rate', quantity='leg1_quantity'),
    'refit':    dict(table='crp_emir_refit_ecb.emir_refit_ecb_trade_states_deduplicated',
                     direction='leg1_direction', country_suffix='_gleif',
                     isin='leg1_product_isin', cfi='leg1_product_cfi',
                     notional='leg1_notional_leg1', ccy='leg1_notional_leg1_currency',
                     price='leg1_price', quantity='leg1_notional_quantity_leg1'),
}

def build_query(regime, region, start, end):
    s = sources[regime]

    # price normalised to a per 100 quotation, kept only if between 50 and 250
    price_norm = f"""CASE WHEN {s['price']} > 25000 THEN {s['price']} / 1000
                        WHEN {s['price']} > 2500  THEN {s['price']} / 100
                        WHEN {s['price']} > 250   THEN {s['price']} / 10
                        ELSE {s['price']} END"""

    # product identification, EA by ISIN list, US by venue and CFI (no contract ISIN on CBOT)
    if region == 'EA':
        product_key    = f"CAST(e.{s['isin']} AS STRING)"
        product_filter = f"e.{s['isin']} IN {euro_govs_isin}"
        product_join   = f"JOIN (\n        {gov_fut_sql}\n    ) g ON e.{s['isin']} = g.isin"
        product_cols   = f"{product_key} AS product_id, g.country AS product_country, g.expiration_date AS expiration_date"
    else:
        product_key    = f"CONCAT('XCBT_FFDPSX_', to_date(e.leg1_maturity_date))"
        product_filter = f"""e.leg1_execution_venue = 'XCBT' AND e.leg1_contract_type = 'FUTR' AND e.leg1_asset_class = 'INTR'
        AND e.{s['cfi']} = 'FFDPSX' AND e.{s['ccy']} = 'USD' AND e.leg1_maturity_date IS NOT NULL"""
        product_join   = ""
        product_cols   = f"{product_key} AS product_id, 'US' AS product_country, to_date(e.leg1_maturity_date) AS expiration_date"

    return f"""
SELECT *
FROM (
    SELECT
        -- identifiers
        CAST(tec_ruti AS STRING)         AS tec_ruti,
        CAST(reference_period AS STRING) AS reference_period,
        -- counterparties
        CAST(CASE WHEN direction = 'BYER' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS STRING) AS buyer_id,
        CAST(CASE WHEN direction = 'BYER' THEN rep_sector             ELSE oth_sector             END AS STRING) AS buyer_sector,
        CAST(CASE WHEN direction = 'BYER' THEN rep_country            ELSE oth_country            END AS STRING) AS buyer_country,
        CAST(CASE WHEN direction = 'SLLR' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS STRING) AS seller_id,
        CAST(CASE WHEN direction = 'SLLR' THEN rep_sector             ELSE oth_sector             END AS STRING) AS seller_sector,
        CAST(CASE WHEN direction = 'SLLR' THEN rep_country            ELSE oth_country            END AS STRING) AS seller_country,
        -- product
        product_id,
        product_country,
        expiration_date,
        -- trade terms, cleaned
        CAST(CASE WHEN quantity IS NULL OR quantity <= 0 OR price_clean IS NULL THEN notional_raw
                  WHEN notional_raw / (quantity * price_clean) > 3000 THEN quantity * 1000 * price_clean
                  ELSE notional_raw END AS DOUBLE) AS notional,
        CAST(notional_currency AS STRING)          AS notional_currency,
        CAST(price_clean AS DOUBLE)                AS price,
        CAST(quantity AS DOUBLE)                   AS quantity,
        CAST(notional_raw AS DOUBLE)               AS notional_raw,
        CAST(price_raw AS DOUBLE)                  AS price_raw
    FROM (
        SELECT
            e.tec_ruti,
            e.reference_period,
            e.{s['direction']} AS direction,
            e.leg1_reporting_cpty_id,
            e.leg1_other_cpty_id,
            e.leg1_reporting_cpty_id_country_code{s['country_suffix']} AS rep_country,
            e.leg1_other_cpty_id_country_code{s['country_suffix']}     AS oth_country,
            CASE WHEN e.leg1_reporting_cpty_id IN {eurex_lei} THEN 'CCP' ELSE s_rep.sector END AS rep_sector,
            CASE WHEN e.leg1_other_cpty_id     IN {eurex_lei} THEN 'CCP' ELSE s_oth.sector END AS oth_sector,
            {product_cols},
            e.{s['notional']} AS notional_raw,
            e.{s['ccy']}      AS notional_currency,
            e.{s['price']}    AS price_raw,
            e.{s['quantity']} AS quantity,
            COALESCE(CASE WHEN ({price_norm}) BETWEEN 50 AND 250 THEN ({price_norm}) END, p.price_ref) AS price_clean
        FROM {s['table']} e
        {product_join}
        LEFT JOIN {sector_table} s_rep ON e.leg1_reporting_cpty_id = s_rep.lei
        LEFT JOIN {sector_table} s_oth ON e.leg1_other_cpty_id = s_oth.lei
        -- reference price: average normalised price of all reporters for the same contract and day
        LEFT JOIN (
            SELECT product_id, reference_period, AVG(pn) AS price_ref
            FROM (
                SELECT {product_key} AS product_id, e.reference_period, {price_norm} AS pn
                FROM {s['table']} e
                WHERE e.reference_period BETWEEN '{start}' AND '{end}' AND {product_filter}
            ) x
            WHERE pn BETWEEN 50 AND 250
            GROUP BY product_id, reference_period
        ) p ON {product_key} = p.product_id AND e.reference_period = p.reference_period
        WHERE e.reference_period BETWEEN '{start}' AND '{end}'
            AND {product_filter}
            AND e.{s['direction']} IN ('BYER', 'SLLR')
            AND e.{s['notional']} > 0 AND e.{s['notional']} <= 1e11
            -- expiry cut off: nothing reported more than 10 days after the contract expired
            AND e.reference_period <= to_date(date_add(CAST({'g.expiration_date' if region == 'EA' else 'e.leg1_maturity_date'} AS TIMESTAMP), 10))
    ) base
) final
-- cap: more than 500,000 implied contracts in one row is a reporting error
WHERE price IS NULL OR notional / (price * 1000) <= 500000
"""

## One day test

In [ ]:
tests = {
    'prerefit_EA': build_query('prerefit', 'EA', '2023-07-17', '2023-07-17'),
    'prerefit_US': build_query('prerefit', 'US', '2023-07-17', '2023-07-17'),
    'refit_EA':    build_query('refit',    'EA', '2025-07-15', '2025-07-15'),
    'refit_US':    build_query('refit',    'US', '2025-07-15', '2025-07-15'),
}
dfs = {k: pd.read_sql_query(q, cnxn) for k, q in tests.items()}
for k, d in dfs.items():
    print(k, d.shape, list(d.columns) == list(dfs['refit_EA'].columns))
dfs['refit_EA'].head()

## Build the table, year by year

In [ ]:
def year_chunks(start, end):
    chunks, y = [], int(start[:4])
    while True:
        a = max(start, f'{y}-01-01')
        b = min(end, f'{y}-12-31')
        chunks.append((a, b))
        if b == end:
            return chunks
        y += 1

def build_table(table, jobs):
    """jobs: list of (label, query_fn(start, end), start, end). CREATE from the first chunk, INSERT INTO for the rest."""
    first = True
    for label, q, start, end in jobs:
        for a, b in year_chunks(start, end):
            t0 = time.time()
            if first:
                cursor.execute(f"DROP TABLE IF EXISTS {table}")
                cursor.execute(f"""
CREATE EXTERNAL TABLE {table}
STORED AS PARQUET TBLPROPERTIES ('external.table.purge'='true')
AS
{q(a, b)}
""")
                first = False
            else:
                cursor.execute(f"INSERT INTO {table}\n{q(a, b)}")
            print(f"{datetime.now():%H:%M:%S}  {label:12s} {a} to {b}  done in {(time.time() - t0) / 60:5.1f} min")

jobs = [
    ('prerefit_EA', lambda a, b: build_query('prerefit', 'EA', a, b), prerefit_start, prerefit_end),
    ('prerefit_US', lambda a, b: build_query('prerefit', 'US', a, b), prerefit_start, prerefit_end),
    ('refit_EA',    lambda a, b: build_query('refit',    'EA', a, b), refit_start,    refit_end),
    ('refit_US',    lambda a, b: build_query('refit',    'US', a, b), refit_start,    refit_end),
]
build_table(table, jobs)

In [ ]:
# check
pd.read_sql_query(f"""
SELECT product_country, MIN(reference_period) AS first_date, MAX(reference_period) AS last_date,
       COUNT(*) AS n_rows, COUNT(DISTINCT product_id) AS n_contracts,
       ROUND(SUM(notional_raw) / 1e12, 1) AS notional_raw_tn, ROUND(SUM(notional) / 1e12, 1) AS notional_tn
FROM {table}
GROUP BY product_country
ORDER BY product_country
""", cnxn)

## Net positions by sector

In [ ]:
query_net = f"""
SELECT reference_period AS bdate, product_country, sector,
       SUM(net) / 1e9 AS net_position_bn
FROM (
    SELECT reference_period, product_country, buyer_sector  AS sector,  notional AS net FROM {table}
    UNION ALL
    SELECT reference_period, product_country, seller_sector AS sector, -notional AS net FROM {table}
) x
GROUP BY reference_period, product_country, sector
ORDER BY reference_period, product_country, sector
"""
df_net = pd.read_sql_query(query_net, cnxn)

for country, d in df_net.groupby('product_country'):
    d.pivot(index='bdate', columns='sector', values='net_position_bn').plot(title=country, figsize=(12, 5))